In [1]:
import pandas as pd
import time
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

def train_and_optimize():
    print("Carregando dataset...")
    # O dataset contém as colunas: hand, WRIST_x, ..., PINKY_TIP_z, label
    df = pd.read_csv("asl_landmarks_dataset.csv")
    
    # 1. Preparar dados
    # Converter a coluna 'hand' (Left/Right) em valores numéricos (0/1) para os modelos
    le_hand = LabelEncoder()
    df['hand'] = le_hand.fit_transform(df['hand'])
    
    # X: Features (hand + 63 coordenadas landmarks)
    # y: Alvo (label A-Z)
    X = df.drop(['label'], axis=1) # Remove apenas a label, mantém a 'hand' como feature numérica
    y = df['label']
    
    # 2. Dividir em treino (70%), validação (15%) e teste (15%)
    # Stratify garante que a distribuição das letras (A-Z) seja mantida em todos os conjuntos
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    # 0.176 de 0.85 é aproximadamente 0.15 do total
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
    )
    
    models = {
        'KNN': {
            'model': KNeighborsClassifier(),
            'params': {
                'n_neighbors': [3, 5, 7],
                'weights': ['uniform', 'distance']
            }
        }
    }
    
    best_model = None
    best_accuracy = 0
    best_inf_time = float('inf')
    results = []

    for name, config in models.items():
        print(f"\nOtimizando {name}...")
        grid = GridSearchCV(config['model'], config['params'], cv=3, n_jobs=-1, verbose=1)
        
        grid.fit(X_train, y_train)
        
        # Avaliação no conjunto de validação
        val_preds = grid.predict(X_val)
        val_acc = accuracy_score(y_val, val_preds)
        
        # Medir tempo de inferência (média de 100 amostras)
        start_inf = time.time()
        for _ in range(100):
            grid.predict(X_val[:1])
        inf_time = (time.time() - start_inf) / 100
        
        print(f"{name} - Val Accuracy: {val_acc:.4f}, Inf Time: {inf_time:.6f}s")
        
        results.append({
            'model_name': name,
            'best_params': grid.best_params_,
            'val_accuracy': val_acc,
            'inf_time': inf_time
        })
        
        # Lógica de seleção do melhor modelo
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            best_inf_time = inf_time
            best_model = grid.best_estimator_
        elif val_acc == best_accuracy:
            # Desempate pelo tempo de inferência mais baixo
            if inf_time < best_inf_time:
                best_inf_time = inf_time
                best_model = grid.best_estimator_

    # 3. Avaliação final no conjunto de teste com o melhor modelo encontrado
    test_preds = best_model.predict(X_test)
    print("\n" + "="*40)
    print("=== Avaliação Final no Conjunto de Teste ===")
    print(f"Melhor Modelo: {best_model.__class__.__name__}")
    print(classification_report(y_test, test_preds))
    
    # 4. Guardar o melhor modelo e o encoder da mão
    with open('melhor_modelo.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    
    # É importante guardar o encoder para saber como processar a 'hand' no futuro
    with open('hand_encoder.pkl', 'wb') as f:
        pickle.dump(le_hand, f)
    
    print(f"Modelo guardado com sucesso.")
    
    # Guardar resultados para documentação
    results_df = pd.DataFrame(results)
    results_df.to_csv('comparacao_modelos.csv', index=False)

if __name__ == "__main__":
    train_and_optimize()

Carregando dataset...

Otimizando KNN...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
KNN - Val Accuracy: 0.9997, Inf Time: 0.003873s

=== Avaliação Final no Conjunto de Teste ===
Melhor Modelo: KNeighborsClassifier
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       150
           B       1.00      1.00      1.00       150
           C       1.00      1.00      1.00       142
           D       1.00      1.00      1.00       150
           E       1.00      1.00      1.00       150
           F       1.00      1.00      1.00       150
           G       1.00      1.00      1.00       149
           H       1.00      1.00      1.00       150
           I       1.00      1.00      1.00       150
           J       1.00      1.00      1.00       150
           K       1.00      1.00      1.00       150
           L       1.00      1.00      1.00       150
           M       1.00      1.00      1.00       141
           N

## Accuracy (Acurácia)

<center>
<span style="font-size: 24px;">
$\text{Accuracy} = \frac{\text{nº de previsões corretas}}{\text{nº total de previsões}}$
</span>
</center>


A métrica principal utilizada foi a acurácia, uma vez que o problema consiste numa classificação multiclasse com classes equilibradas, onde todas as letras têm igual importância.

        'RandomForest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {'n_estimators': [100, 200], 'max_depth': [None, 20]}
        },
        'KNN': {
            'model': KNeighborsClassifier(),
            'params': {'n_neighbors': [3, 5], 'weights': ['distance']}
        },
        'SVM': {
            'model': SVC(probability=True, random_state=42),
            'params': {'C': [1, 10], 'kernel': ['rbf']}
        },
        'DecisionTree': {
            'model': DecisionTreeClassifier(random_state=42),
            'params': {'max_depth': [None, 10, 30]}
        },
        'NeuralNetwork': {
            'model': MLPClassifier(random_state=42, max_iter=1000),
            'params': {'hidden_layer_sizes': [(128, 64), (64, 64)], 'activation': ['relu']}
        },